# Data Definition

## Fetching the Data

In [ ]:
import pathlib

!rm -rf sample_data

if not pathlib.Path("landscape").is_dir():
  !wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape

!mkdir -p to-annotate

ids = [10034, 10159, 10244, 1060, 1072, 11, 11105, 1223, 12326, 12971]
for i, id in enumerate(ids):
  !cp landscape/seg_pred/no_supervision/{id}.jpg to-annotate/{i}.jpg

## Writing Labeling Instructions

By studying the data in `landscape/seg_train`, write labeling instructions so that, looking at an image, an annotator knows which of the 6 classes to assign: `buildings`, `forest`, `glacier`, `mountain`, `sea` and `street`.

The next cell shows a few random images of each class.

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image

# Six random images of each class of the training set
random.seed(0)
classes = sorted(path.name for path in pathlib.Path("landscape/seg_train").iterdir())
fig, axes = plt.subplots(len(classes), 6, figsize=(12, 2.2 * len(classes)))
for row, name in zip(axes, classes):
  paths = random.sample(sorted(pathlib.Path("landscape/seg_train", name).glob("*.jpg")), 6)
  for ax, path in zip(row, paths):
    ax.imshow(Image.open(path))
    ax.axis("off")
  row[0].set_title(name, loc="left", fontweight="bold")
plt.tight_layout()

*Your instructions*

## Labeling

Use someone else's labeling instructions, following them as closely as possible, to annotate the images in the `to-annotate` folder.

Fill the `labels` list with the classes you decided to assign to each of the images.

The 10 images to annotate, numbered 0 to 9 in the order `labels` must list them:

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.flat, range(10)):
  ax.imshow(Image.open(f"to-annotate/{i}.jpg"))
  ax.set_title(f"Image {i}")
  ax.axis("off")
plt.tight_layout()

In [ ]:
num_classes = 6
class_to_index = dict(buildings=0, forest=1, glacier=2, mountain=3, sea=4, street=5)

labels = []  # To fill

## Computing inter-rater agreement

Pool your annotations with the other participants', on whatever the trainer sets up (a shared spreadsheet or a form, for instance), then use Fleiss' kappa to compute the inter-annotator agreement. It's an extension of Cohen's kappa to the case of more than two annotators. It measures the agreement between annotators:

- The score is between -1 and 1
- 0 is an agreement matching chance
- Below 0, annotators agree less than a random draw of the classes would
- Conversely above
- The interpretation of this score depends on the number of classes, and is preferably relative rather than absolute

You can compute it with the [`statsmodels.stats.inter_rater.fleiss_kappa`](https://www.statsmodels.org/stable/generated/statsmodels.stats.inter_rater.fleiss_kappa.html) method, or by implementing the computations yourself (detailed for instance [on Wikipedia](https://en.wikipedia.org/wiki/Fleiss%27_kappa)).

In [ ]:
#  Your code here

### Solution

In [ ]:
import pathlib

import numpy
import statsmodels.stats.inter_rater


def one_hot_encode(answers, num_classes=num_classes):
  return numpy.eye(num_classes)[answers]


all_labels = [
  # One list per annotator, in image order (yours is `labels`).
  # Example, to replace with the group's annotations:
  ["glacier", "buildings", "sea", "mountain", "sea", "glacier", "forest", "mountain", "glacier", "sea"],
  ["glacier", "street", "mountain", "glacier", "sea", "mountain", "street", "mountain", "glacier", "street"],
  ["sea", "buildings", "mountain", "glacier", "street", "mountain", "forest", "glacier", "glacier", "sea"],
]

n_files = len(list(pathlib.Path("to-annotate").glob("*.jpg")))

total = numpy.zeros((n_files, num_classes))
for answer in all_labels:
  total += one_hot_encode([class_to_index[name] for name in answer])

statsmodels.stats.inter_rater.fleiss_kappa(total)

## Impact on Human Level Performance

Human Level Performance (*HLP*) is widely used in Machine Learning. What's the impact of bad labeling on HLP? Is a learning algorithm affected by bad labels in the same way?

*Your answer here*

### Solution

Some answer elements:

- HLP is measured by comparing a human's labels with a reference (or with the other annotators'). With an ambiguous guide, competent annotators contradict each other: the measured HLP drops, but it then reflects inconsistent labels more than how hard the task really is. So it is a poor estimate of the Bayes error.
- A model can then seem to "beat human level" just because it reproduces the majority label better. That's no real progress: clarifying the guide raises HLP and shows the margin left.
- A learning algorithm isn't affected by every error in the same way:
  - few, random errors partly cancel out over many examples, but they call for more data and lower the reachable performance;
  - systematic errors (for instance, always calling a snowy mountain `glacier`) are learned as they are: the model faithfully reproduces the annotators' implicit rule;
  - noisy labels in the test set make the evaluation itself unreliable.
- Making labeling more consistent (the guide, double labeling of ambiguous cases) often pays off more than changing the model.